# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [31]:
# Write your code below.

%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [39]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [37]:
import os
from glob import glob

PRICE_DATA = os.getenv('PRICE_DATA')

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)

print(f"Found {len(parquet_files)} parquet files.")
for file in parquet_files:
    print(file)


Found 3328 parquet files.
../../05_src/data/prices\AAPL\AAPL_2000\part.0.parquet
../../05_src/data/prices\AAPL\AAPL_2000\part.1.parquet
../../05_src/data/prices\AAPL\AAPL_2001\part.0.parquet
../../05_src/data/prices\AAPL\AAPL_2001\part.1.parquet
../../05_src/data/prices\AAPL\AAPL_2002\part.0.parquet
../../05_src/data/prices\AAPL\AAPL_2002\part.1.parquet
../../05_src/data/prices\AAPL\AAPL_2003\part.0.parquet
../../05_src/data/prices\AAPL\AAPL_2003\part.1.parquet
../../05_src/data/prices\AAPL\AAPL_2004\part.0.parquet
../../05_src/data/prices\AAPL\AAPL_2004\part.1.parquet
../../05_src/data/prices\AAPL\AAPL_2005\part.0.parquet
../../05_src/data/prices\AAPL\AAPL_2005\part.1.parquet
../../05_src/data/prices\AAPL\AAPL_2006\part.0.parquet
../../05_src/data/prices\AAPL\AAPL_2006\part.1.parquet
../../05_src/data/prices\AAPL\AAPL_2007\part.0.parquet
../../05_src/data/prices\AAPL\AAPL_2007\part.1.parquet
../../05_src/data/prices\AAPL\AAPL_2008\part.0.parquet
../../05_src/data/prices\AAPL\AAPL_2008

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [55]:
ddf = dd.read_parquet(parquet_files).set_index("Ticker")

print(len(ddf))

dd_shift = ddf.groupby('Ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)

dd_feat= dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)

dd_feat = dd_feat.assign(
    hi_lo_range = lambda x: x['High'] - x['Low']
)

dd_feat


403456


C:\Users\rossm\AppData\Local\Temp\ipykernel_9948\3476136450.py:5: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = ddf.groupby('Ticker', group_keys=False).apply(


,Date,Close,High,Low,Open,Volume,Year,Close_lag_1,Returns,hi_lo_range
npartitions=64,,,,,,,,,,
AAPL,datetime64[ns],float64,float64,float64,float64,float64,int32,float64,float64,float64
ACN,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
ZBRA,...,...,...,...,...,...,...,...,...,...
ZBRA,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [51]:
pdf = dd_feat.compute()

pdf = pdf.assign(
    moving_avg_10 = lambda x: x["Returns"].rolling(window=10).mean()
)

# pdf["moving_avg_10"] = pdf["Returns"].rolling(window=10).mean()

pdf.head(15)

,Date,Close,High,Low,Open,Volume,Year,Close_lag_1,Returns,hi_lo_range,mean_avg_10,moving_avg_10
Ticker,,,,,,,,,,,,
AAPL,2000-01-03,0.843077,0.847313,0.765877,0.789884,5.357968e+08,2000,NaN,NaN,0.081436,NaN,NaN
AAPL,2000-01-04,0.771997,0.833191,0.762111,0.815303,5.123776e+08,2000,0.843077,-0.084310,0.071080,NaN,NaN
AAPL,2000-01-05,0.783293,0.832720,0.775762,0.781411,7.783216e+08,2000,0.771997,0.014633,0.056958,NaN,NaN
AAPL,2000-01-06,0.715509,0.805889,0.715509,0.799299,7.679728e+08,2000,0.783293,-0.086538,0.090380,NaN,NaN
AAPL,2000-01-07,0.749401,0.760699,0.719275,0.726806,4.607344e+08,2000,0.715509,0.047369,0.041424,NaN,NaN
AAPL,2000-01-10,0.736221,0.770113,0.713626,0.768230,5.050640e+08,2000,0.749401,-0.017588,0.056487,NaN,NaN
AAPL,2000-01-11,0.698563,0.748460,0.681617,0.722570,4.415488e+08,2000,0.736221,-0.051151,0.066844,NaN,NaN
AAPL,2000-01-12,0.656668,0.719275,0.651489,0.715509,9.760688e+08,2000,0.698563,-0.059973,0.067786,NaN,NaN
AAPL,2000-01-13,0.728689,0.743752,0.696680,0.711625,1.032685e+09,2000,0.656668,0.109677,0.047072,NaN,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [52]:
dd_feat = dd_feat.assign(
    mean_avg_10 = lambda x: x['Returns'].rolling(window=10).mean()
)

dd_feat.head(15)

,Date,Close,High,Low,Open,Volume,Year,Close_lag_1,Returns,hi_lo_range,mean_avg_10
Ticker,,,,,,,,,,,
AAPL,2000-01-03,0.843077,0.847313,0.765877,0.789884,5.357968e+08,2000,NaN,NaN,0.081436,NaN
AAPL,2000-01-04,0.771997,0.833191,0.762111,0.815303,5.123776e+08,2000,0.843077,-0.084310,0.071080,NaN
AAPL,2000-01-05,0.783293,0.832720,0.775762,0.781411,7.783216e+08,2000,0.771997,0.014633,0.056958,NaN
AAPL,2000-01-06,0.715509,0.805889,0.715509,0.799299,7.679728e+08,2000,0.783293,-0.086538,0.090380,NaN
AAPL,2000-01-07,0.749401,0.760699,0.719275,0.726806,4.607344e+08,2000,0.715509,0.047369,0.041424,NaN
AAPL,2000-01-10,0.736221,0.770113,0.713626,0.768230,5.050640e+08,2000,0.749401,-0.017588,0.056487,NaN
AAPL,2000-01-11,0.698563,0.748460,0.681617,0.722570,4.415488e+08,2000,0.736221,-0.051151,0.066844,NaN
AAPL,2000-01-12,0.656668,0.719275,0.651489,0.715509,9.760688e+08,2000,0.698563,-0.059973,0.067786,NaN
AAPL,2000-01-13,0.728689,0.743752,0.696680,0.711625,1.032685e+09,2000,0.656668,0.109677,0.047072,NaN


In [53]:
len(dd_feat)

403456

Pandas Was not Necessary to calculate the moving average return. As seen above I have done it in dask.

I think it would be better to do it into dask as then there is no reason to load the full 403456 rows into memory and you are able to manipulate the data quicker.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.